# 🧪 Sesión 16 · OAuth2 Password Flow y Header Authorization Bearer

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué el estándar OAuth2 usa un formulario y no JSON para el login?

---

## 🖥️ Así se verá el resultado final

```http
POST /token
Content-Type: application/x-www-form-urlencoded
username=ana&password=secreta
HTTP/1.1 200 OK
{"access_token":"...","token_type":"bearer"}

GET /api/perfil   Authorization: Bearer ...
HTTP/1.1 200 OK
{"usuario":"ana"}

GET /api/perfil   (sin token)
HTTP/1.1 401 Unauthorized
WWW-Authenticate: Bearer
```

_El formulario y el encabezado no son un capricho: son el contrato que esperan los clientes OAuth2._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar el Password Flow de OAuth2 con FastAPI.
- Enviar y validar el encabezado Authorization: Bearer.
- Responder 401 con el reto WWW-Authenticate correcto.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El formulario de login emite token → HTTP 200
- [ ] El token Bearer abre el perfil → HTTP 200
- [ ] Sin token se responde 401 con WWW-Authenticate → HTTP 401
- [ ] Un esquema distinto se rechaza → HTTP 401
- [ ] Se usa el flujo OAuth2 de FastAPI

## 🧩 El reto

Implementa el Password Flow de OAuth2 con formulario y protege una ruta con OAuth2PasswordBearer.

**Requisitos del contrato**

- El usuario de prueba es ana con contraseña secreta.
- POST /token recibe username y password como formulario, no como JSON.
- Responde 200 con access_token y token_type bearer.
- GET /api/perfil exige Authorization: Bearer y responde 200 con el usuario.
- Sin token responde 401 e incluye WWW-Authenticate: Bearer.
- Un esquema distinto de Bearer también responde 401.

> **Pista conceptual:** `OAuth2PasswordRequestForm` se encarga del formulario; `OAuth2PasswordBearer` se encarga del encabezado y del 401 con el reto correcto.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion16.md` con la diferencia entre el formulario y el encabezado.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm

app = FastAPI(title="OAuth2 Password Flow")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.

# OAuth2 espera obtener el token desde /token
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

USUARIO = "ana"
PASSWORD = "secreta"


@app.post("/token")
async def login(form_data: OAuth2PasswordRequestForm = Depends()):
    if form_data.username == USUARIO and form_data.password == PASSWORD:
        return {
            "access_token": USUARIO,
            "token_type": "bearer"
        }

    raise HTTPException(
        status_code=status.HTTP_401_UNAUTHORIZED,
        detail="Credenciales incorrectas"
    )


@app.get("/api/perfil")
async def perfil(token: str = Depends(oauth2_scheme)):
    if token != USUARIO:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Token inválido",
            headers={"WWW-Authenticate": "Bearer"},
        )

    return {"usuario": USUARIO}


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Aplicación en FastAPI ejecutada en Google Colab. Se debe implementar OAuth2 Password Flow usando formulario para el login y token Bearer para proteger rutas. |
| **Rol** | Desarrollador backend encargado de autenticar usuarios, generar tokens y proteger recursos. |
| **Especificación** | Crear POST /token que reciba username y password por formulario. El usuario válido es ana y la contraseña secreta. Debe devolver un token Bearer. Crear GET /api/perfil protegido con OAuth2. Con token válido responde el usuario; sin token o con otro esquema responde 401 con WWW-Authenticate: Bearer. |
| **Output** | Archivo app.py funcional usando OAuth2PasswordRequestForm y OAuth2PasswordBearer, cumpliendo todas las pruebas del laboratorio. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'El formulario de login emite token',
  'metodo': 'POST',
  'ruta': '/token',
  'data': {'username': 'ana', 'password': 'secreta'},
  'estado': 200,
  'json_contiene': {'token_type': 'bearer'},
  'guardar': {'token': 'access_token'}},
 {'descripcion': 'El token Bearer abre el perfil',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token}'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana'}},
 {'descripcion': 'Sin token se responde 401 con WWW-Authenticate',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'estado': 401,
  'headers_contiene': {'www-authenticate': 'bearer'}},
 {'descripcion': 'Un esquema distinto se rechaza',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Basic YW5hOnNlY3JldGE='},
  'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. El formulario de login emite token
✅ 02. El token Bearer abre el perfil
✅ 03. Sin token se responde 401 con WWW-Authenticate
✅ 04. Un esquema distinto se rechaza
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


In [4]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["OAuth2PasswordBearer", "OAuth2PasswordRequestForm"], descripcion="Se usa el flujo OAuth2 de FastAPI"))
resumen(resultados)


✅ Se usa el flujo OAuth2 de FastAPI
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion16.md` a `auditorias/auditoria_sesion16.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue entender cómo funcionaba el token Bearer, porque al principio pensaba que solo bastaba con recibir usuario y contraseña. Después comprendí que primero se genera un token y luego ese token se utiliza para acceder a las rutas protegidas.

   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí usar el nombre del usuario (ana) como valor del token para mantener la solución simple y fácil de entender, ya que el contrato no obligaba a generar un JWT real.

   - ¿Qué error detectó la verificación que tú no habías previsto?
   La verificación comprobó que cuando no se envía el token, la respuesta debe ser 401 y además incluir el encabezado WWW-Authenticate: Bearer. Al principio no sabía que ese encabezado también era parte importante del contrato.
   
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
